# Stg Layer — places (Google Maps Text Search results, full rebuild)

Source: `bronze.places_raw` (Lakebase `public.places`, written by the search job and by `auto_search` mode `places`), `bronze.place_search_results_raw` and `bronze.search_keywords_raw`. Added 2026-10-31 (auto search on Google Maps).

1. Latest Bronze read per `place_id`.
2. Keep **food places that are open**: a Google type in `FOOD_TYPES` or ending in `_restaurant` (same rule as `stg/restaurant_geocode`), `business_status` NULL or `OPERATIONAL`.
3. `ward` = second-to-last part of `formatted_address` (as geocode); `city` from the address (HCMC → the city label geocode uses).
4. `district` (approximate ❓): Google returns no district (post-2025 wards), so it is the district of the grid keyword (`search_keywords.district`) that found the place most often; NULL when only non-grid keywords found it.
5. `restaurant_name_normalized` = same rule as `stg_restaurant_mentions`.

`website_uri` / `phone_number` / `google_opening_hours` (Google's weekday lines joined with `; `) and the rating / price columns are **Enterprise** fields: NULL for places that only `auto_search` found (Text Search **Pro** since 2026-11-01) until `stg_restaurant_place_details` reads them. `search_hits` = distinct keywords that found the place (Place Details reads the most-found Google-only places first).

Columns use the names of `stg_restaurant_place_details` so `gold/restaurants` can read both the same way. Small table (≤ ~1,000 places / month): full `CREATE OR REPLACE` each run.

In [ ]:
%run ../gold/_udfs

In [ ]:
# --- Config ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

PLACES_RAW = f"{BRONZE}.places_raw"
HITS_RAW = f"{BRONZE}.place_search_results_raw"
KEYWORDS_RAW = f"{BRONZE}.search_keywords_raw"
GEOCODE = f"{SILVER}.stg_restaurant_geocode"
TARGET = f"{SILVER}.stg_places"
FOOD_TYPES = ["restaurant", "food", "cafe", "coffee_shop", "bakery", "bar", "meal_takeaway", "meal_delivery",
              "food_court", "dessert_shop", "ice_cream_shop", "tea_house", "juice_shop", "snack_bar", "diner",
              "bistro", "deli", "pub"]
print(f"Bronze source: {PLACES_RAW} · Silver target: {TARGET}")

In [ ]:
# --- Rebuild stg_places (empty table until the first Places search reaches Bronze) ---
hits_sql = f"""
    SELECT h.place_id, k.district, COUNT(*) AS hits, MIN(h.fetched_at) AS first_hit
    FROM (SELECT DISTINCT id, place_id, search_keyword, fetched_at FROM {HITS_RAW}) h
    JOIN (SELECT keyword_normalized, max_by(district, ingest_ts) AS district
          FROM {KEYWORDS_RAW} WHERE mode = 'places' GROUP BY keyword_normalized) k
      ON k.keyword_normalized = normalize_keyword(h.search_keyword)
    WHERE k.district IS NOT NULL
    GROUP BY h.place_id, k.district
""" if spark.catalog.tableExists(HITS_RAW) and spark.catalog.tableExists(KEYWORDS_RAW) else \
    "SELECT CAST(NULL AS STRING) AS place_id, CAST(NULL AS STRING) AS district, 0 AS hits, CAST(NULL AS TIMESTAMP) AS first_hit WHERE false"
# How many distinct keywords found each place (any keyword): Place Details reads the most-found Google-only places first
search_hits_sql = f"""
    SELECT place_id, COUNT(DISTINCT normalize_keyword(search_keyword)) AS search_hits
    FROM {HITS_RAW} GROUP BY place_id
""" if spark.catalog.tableExists(HITS_RAW) else "SELECT CAST(NULL AS STRING) AS place_id, 0 AS search_hits WHERE false"
hcmc_label = "Hồ Chí Minh"
if spark.catalog.tableExists(GEOCODE):
    row = spark.sql(f"SELECT max(city) AS c FROM {GEOCODE} WHERE unaccent_vi(lower(city)) LIKE '%ho chi minh%'").collect()[0]
    hcmc_label = row["c"] or hcmc_label
food = ", ".join(f"'{t}'" for t in FOOD_TYPES)

if spark.catalog.tableExists(PLACES_RAW):
    spark.sql(f"""
        CREATE OR REPLACE TABLE {TARGET}
        COMMENT 'Google Maps places found by Text Search (search job / auto_search), food places that are open. Latest read per place_id.'
        AS
        WITH latest AS (
            SELECT * FROM (
                SELECT *, ROW_NUMBER() OVER (PARTITION BY place_id ORDER BY COALESCE(updated_at, created_at) DESC, ingest_ts DESC) AS rn
                FROM {PLACES_RAW}
            ) WHERE rn = 1
        ),
        district_guess AS (
            SELECT place_id, max_by(district, struct(hits, -unix_timestamp(first_hit))) AS district
            FROM ({hits_sql}) GROUP BY place_id
        ),
        search_hits AS ({search_hits_sql}),
        parts AS (
            SELECT l.*, transform(split(formatted_address, ','), x -> trim(x)) AS addr_parts FROM latest l
        )
        SELECT
            p.place_id,
            p.display_name                                              AS place_name,
            regexp_replace(lower(trim(unaccent_vi(normalize_keyword(p.display_name)))), '[^a-z0-9 ]', '') AS restaurant_name_normalized,
            p.formatted_address,
            CASE WHEN size(p.addr_parts) >= 3 THEN element_at(p.addr_parts, -2) END AS ward,
            d.district,
            CASE WHEN unaccent_vi(lower(p.formatted_address)) LIKE '%ho chi minh%' THEN '{hcmc_label}'
                 ELSE element_at(p.addr_parts, -1) END                 AS city,
            CAST(p.latitude AS DOUBLE)                                  AS latitude,
            CAST(p.longitude AS DOUBLE)                                 AS longitude,
            p.primary_type,
            CAST(p.types AS ARRAY<STRING>)                              AS place_types,
            CAST(p.rating AS DOUBLE)                                    AS rating,
            CAST(p.user_rating_count AS INT)                            AS rating_count,
            p.price_level,
            CAST(p.price_range_start_vnd AS BIGINT)                     AS price_start_vnd,
            CAST(p.price_range_end_vnd AS BIGINT)                       AS price_end_vnd,
            p.business_status,
            p.google_maps_uri,
            -- Enterprise fields: NULL for places only found by auto_search (Text Search Pro) until Place Details reads them
            p.website_uri,
            p.national_phone_number                                     AS phone_number,
            nullif(concat_ws('; ', from_json(get_json_object(CAST(p.regular_opening_hours AS STRING), '$.weekdayDescriptions'),
                                             'array<string>')), '')  AS google_opening_hours,
            COALESCE(h.search_hits, 0)                                  AS search_hits,
            p.source,
            COALESCE(p.updated_at, p.created_at)                        AS fetched_at,
            current_timestamp()                                         AS etl_date
        FROM parts p
        LEFT JOIN district_guess d ON d.place_id = p.place_id
        LEFT JOIN search_hits h ON h.place_id = p.place_id
        WHERE (p.business_status IS NULL OR p.business_status = 'OPERATIONAL')
          AND (arrays_overlap(CAST(p.types AS ARRAY<STRING>), array({food}))
               OR exists(CAST(p.types AS ARRAY<STRING>), t -> endswith(t, '_restaurant')))
    """)
else:
    print(f"{PLACES_RAW} does not exist yet (no Places search in Bronze): stg_places stays empty")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {TARGET} (
            place_id STRING, place_name STRING, restaurant_name_normalized STRING, formatted_address STRING,
            ward STRING, district STRING, city STRING, latitude DOUBLE, longitude DOUBLE, primary_type STRING,
            place_types ARRAY<STRING>, rating DOUBLE, rating_count INT, price_level STRING, price_start_vnd BIGINT,
            price_end_vnd BIGINT, business_status STRING, google_maps_uri STRING, website_uri STRING,
            phone_number STRING, google_opening_hours STRING, search_hits INT, source STRING,
            fetched_at TIMESTAMP, etl_date TIMESTAMP
        ) USING DELTA
    """)

s = spark.sql(f"SELECT COUNT(*) AS n, COUNT(district) AS with_district, COUNT(rating) AS with_rating FROM {TARGET}").collect()[0]
total = spark.table(PLACES_RAW).select("place_id").distinct().count() if spark.catalog.tableExists(PLACES_RAW) else 0
print(f"✅ {TARGET}: {s['n']} food places (of {total} in Bronze) · {s['with_district']} with a district · {s['with_rating']} with a rating")
display(spark.sql(f"SELECT district, COUNT(*) AS n FROM {TARGET} GROUP BY 1 ORDER BY 2 DESC"))
print("\n--- stg_places complete ---")